<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# ERM, Maximum Likelihood, NLL, KL, and Cross-Entropy

## Motivation

Many machine-learning methods are introduced as empirical risk minimization problems:

$$
\hat{\theta}=\arg\min_\theta \frac{1}{N}\sum_{n=1}^N \ell(y_n,s_\theta(x_n)).
$$

This form is deliberately general. A model produces a score or prediction $s_\theta(x)$, a loss $\ell$ assigns a cost to that prediction, and training chooses the parameter value $\theta$ with small average loss.

The loss, however, is not only an optimization device. Different losses encode different modeling assumptions. Squared error, logistic cross-entropy, hinge loss, and absolute error are all possible ERM choices, but they do not all have the same statistical interpretation.

This lecture focuses on an important special case. If the model specifies a conditional distribution $p_\theta(y\mid x)$ and the ERM loss is chosen to be

$$
\ell(y,s_\theta(x))=-\log p_\theta(y\mid x),
$$

**then ERM coincides with conditional maximum likelihood.** Consequently, minimizing empirical risk can be interpreted as choosing parameters that assign high probability to the observed labels.

Here, $\theta$ denotes the adjustable quantities defining the model. In linear regression, $\theta$ might be the coefficient vector $w$. In logistic regression, it might again be $w$. In a neural network, $\theta$ would include all weights and biases.

The lecture develops this connection in three steps. First, conditional MLE is rewritten as ERM with NLL loss. Second, Gaussian and Bernoulli models show how squared error and logistic cross-entropy arise from likelihoods. Third, the population version of NLL is identified as cross-entropy, or equivalently KL divergence.


## Conditional Modeling in Supervised Learning

We observe training data

$$
\{(x_n,y_n)\}_{n=1}^N.
$$

A typical modeling assumption for supervised-learning model factors the joint distribution as

$$
p_\theta(x,y)=p_\theta(y\mid x)p(x).
$$

Here, $\theta$ is short-hand for all of the model parameters. For example:

- In linear regression, $\theta=w$, and $w$ determines the conditional mean $w^\top x$.
- In logistic regression, $\theta=w$, and $w$ determines the conditional probability $\sigma(w^\top x)$.
- In a neural network, $\theta$ is the collection of all weights and biases, and those determine the output probabilities or predictions.

The factorization separates the part we model from the part we do not model:

- $p_\theta(y\mid x)$ is the conditional model we want to learn.
- $p(x)$ is the marginal distribution of inputs.
- $p(x)$ does **not** depend on $\theta$.

This is the first important simplification. In supervised learning, we often do not model how the covariates themselves are generated. We treat the observed inputs as given and focus on how $Y$ behaves conditional on $X=x$.

Learning means choosing a value of $\theta$, and therefore choosing one conditional distribution from the family $\{p_\theta(y\mid x):\theta\in\Theta\}$.


## Maximum Likelihood for Conditional Models

Maximum likelihood estimation chooses the parameter value that makes the observed data most probable under the model family. MLE chooses

$$
\hat{\theta}
=\arg\max_\theta p_\theta(x_1,\ldots,x_N,y_1,\ldots,y_N).
$$

Assuming independent observations, this becomes

$$
\hat{\theta}
=\arg\max_\theta \prod_{n=1}^N p_\theta(x_n,y_n).
$$

Using the conditional factorization,

$$
\prod_{n=1}^N p_\theta(x_n,y_n)
=\prod_{n=1}^N p_\theta(y_n\mid x_n)p(x_n).
$$

Since $p(x_n)$ does not depend on $\theta$, those factors do not affect which $\theta$ maximizes the objective. Therefore,

$$
\hat{\theta}
=\arg\max_\theta \prod_{n=1}^N p_\theta(y_n\mid x_n).
$$

The supervised likelihood therefore depends on the conditional model for labels given inputs, not on the marginal distribution of the inputs.


## From Likelihood to Negative Log-Likelihood

We can now rewrite the conditional likelihood in a form that looks like an optimization objective from machine learning.

The logarithm is strictly increasing, so taking logs does not change the maximizer:

$$
\hat\theta =\arg\max_\theta \sum_{n=1}^N \log p_\theta(y_n\mid x_n).
$$

Multiplication by $-1$ reverses ordering, so maximizing log-likelihood is equivalent to minimizing negative log-likelihood:

$$
\hat\theta = \arg\min_\theta \sum_{n=1}^N -\log p_\theta(y_n\mid x_n).
$$

Finally, dividing by $N$ does not change the optimizer. It simply expresses the objective as an average:

$$
\hat\theta = \arg\min_\theta \frac{1}{N}\sum_{n=1}^N -\log p_\theta(y_n\mid x_n).
$$


## From MLE to an ERM Problem

The previous expression has the form of an empirical risk minimization problem because it averages one loss value per training example.

The usual ERM template is

$$
\hat{\theta}
=\arg\min_\theta \frac{1}{N}\sum_{n=1}^N \ell(y_n,s_\theta(x_n)),
$$

where:

- $s_\theta(x)$ is the model output or score computed from $x$.
- $\ell(y,s_\theta(x))$ measures how costly that output is when the observed response is $y$.

If the score $s_\theta(x)$ defines a conditional distribution $p_\theta(y\mid x)$, and if we choose the ERM loss to be

$$
\ell(y,s_\theta(x))=-\log p_\theta(y\mid x),
$$

then the ERM objective is the average negative log-likelihood.

This is the central bridge. ERM emphasizes optimization: choose a score, choose a loss, and minimize the training average. MLE emphasizes modeling: choose a probability model and fit the parameter that makes the observed data likely. When the ERM loss is NLL, these are the same optimization problem written in two languages.

Different conditional probability models are equivalent to different NLL losses. The next two examples show two standard cases:

- Gaussian conditional model $\Rightarrow$ squared error.
- Bernoulli conditional model $\Rightarrow$ logistic cross-entropy.


## Example 1: Linear Regression from a Gaussian Model

Assume the conditional model

$$
y_n\mid x_n\sim \mathcal{N}(w^\top x_n,\sigma^2),
$$

where $\sigma^2$ is fixed. Here the general parameter $\theta$ is the regression coefficient vector $w$. The conditional density is

$$
p_w(y_n\mid x_n)
=\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left(-\frac{(y_n-w^\top x_n)^2}{2\sigma^2}\right).
$$

Taking the negative log gives

$$
-\log p_w(y_n\mid x_n)
=\frac{(y_n-w^\top x_n)^2}{2\sigma^2}
+\frac{1}{2}\log(2\pi\sigma^2).
$$

The second term does not depend on $w$. Since $\sigma^2$ is fixed, the factor $1/(2\sigma^2)$ also does not affect the optimizer. Therefore,

$$
\arg\min_w \frac{1}{N}\sum_{n=1}^N -\log p_w(y_n\mid x_n)
=\arg\min_w \frac{1}{N}\sum_{n=1}^N (y_n-w^\top x_n)^2.
$$

Thus, **under this Gaussian conditional model, squared-error and linear-score ERM is the same as the MLE estimate of $w$.**


### Simulating the Gaussian Model

The Gaussian model says that each response is a noisy draw around the conditional mean plane $w^\top x$. This gives a direct way to simulate data from the model.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

N = 100
d = 3

w_true = np.array([-1.0, 2.0, 5.0])
sigma = 1.0

X = rng.normal(size=(N, d - 1))
X = np.hstack([np.ones((N, 1)), X])

s = X @ w_true
y = rng.normal(loc=s, scale=sigma)


In [ ]:
import plotly.graph_objects as go

x2 = X[:, 1]
x3 = X[:, 2]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=x2,
    y=x3,
    z=y,
    mode="markers",
    marker=dict(size=3),
    name="observed y"
))

x2_grid, x3_grid = np.meshgrid(
    np.linspace(x2.min(), x2.max(), 20),
    np.linspace(x3.min(), x3.max(), 20)
)

y_plane = w_true[0] + w_true[1] * x2_grid + w_true[2] * x3_grid

fig.add_trace(go.Surface(
    x=x2_grid,
    y=x3_grid,
    z=y_plane,
    opacity=0.5,
    name="conditional mean"
))

fig.update_layout(
    scene=dict(xaxis_title="x2", yaxis_title="x3", zaxis_title="y"),
    title="Gaussian regression: noisy observations around a mean plane"
)

fig.show()


## Example 2: Logistic Regression from a Bernoulli Model

For binary labels, assume a probabilistic model:

$$
y_n\mid x_n\sim \operatorname{Bernoulli}(p_n),
$$

with

$$
p_n=\sigma(w^\top x_n),
\qquad
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

Again, the general parameter $\theta$ is represented here by the coefficient vector $w$. The model says

$$
P(y_n=1\mid x_n)=\sigma(w^\top x_n).
$$

The Bernoulli likelihood is

$$
p_w(y_n\mid x_n)
=\sigma(w^\top x_n)^{y_n}
\left(1-\sigma(w^\top x_n)\right)^{1-y_n}.
$$

Taking the negative log gives

$$
-\log p_w(y_n\mid x_n)
=-\left[
 y_n\log\sigma(w^\top x_n)
 +(1-y_n)\log(1-\sigma(w^\top x_n))
\right].
$$

From a ERM perspective, we get the same optimization problem by letting $s=w^\top x$ and using a cross-entropy loss:
$$
\ell(y,s)
=-\left[y\log\sigma(s)+(1-y)\log(1-\sigma(s))\right].
$$

Consequently, **a Bernoulli conditional model MLE and the logistic cross-entropy ERM give the same estimate of $w$.**


### Simulating the Bernoulli Model

The Bernoulli model first converts the linear score $w^\top x$ into a probability, then draws a binary label from that probability. The decision boundary is the set of points where $w^\top x=0$, equivalently $P(Y=1\mid X=x)=0.5$.


In [ ]:
rng = np.random.default_rng(11)

N = 200
d = 3

w_true = np.array([-1.0, 2.0, 5.0])
X = rng.normal(size=(N, d - 1))
X = np.hstack([np.ones((N, 1)), X])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

s = X @ w_true
p = sigmoid(s)
y = rng.binomial(1, p)

y[:10]


In [ ]:
x2 = X[:, 1]
x3 = X[:, 2]

x2_grid, x3_grid = np.meshgrid(
    np.linspace(x2.min(), x2.max(), 200),
    np.linspace(x3.min(), x3.max(), 200)
)
score_grid = w_true[0] + w_true[1] * x2_grid + w_true[2] * x3_grid
prob_grid = sigmoid(score_grid)

plt.figure(figsize=(7, 5))
plt.contourf(x2_grid, x3_grid, prob_grid, levels=20, cmap="RdBu_r", alpha=0.65)
plt.colorbar(label=r"$P(Y=1\mid X=x)$")
plt.contour(x2_grid, x3_grid, score_grid, levels=[0], colors="black", linewidths=2)
plt.scatter(x2[y == 0], x3[y == 0], s=20, label="y = 0", alpha=0.8)
plt.scatter(x2[y == 1], x3[y == 1], s=20, label="y = 1", alpha=0.8)
plt.xlabel("x2")
plt.ylabel("x3")
plt.title("Logistic regression: probabilities, labels, and the decision boundary")
plt.legend()
plt.show()


## From Training NLL to Population Log-Loss

So far, the connection has been finite-sample: conditional MLE can be written as ERM when the loss is NLL. The same objective also has a population interpretation.

The empirical NLL is

$$
\frac{1}{N}\sum_{n=1}^N -\log p_\theta(y_n\mid x_n).
$$

The corresponding population objective is

$$
\mathbb{E}_{(X,Y)\sim p}\left[-\log p_\theta(Y\mid X)\right].
$$

This is the expected log-loss on future data. The future label $Y$ is generated from the true distribution $p$, but it is scored using the fitted model $p_\theta$.

This reframes the question. We are no longer asking only which parameter makes the training labels likely. We are asking what average log-loss the model would incur under the true data-generating process.


## Conditional Cross-Entropy

To interpret the population objective, first fix an input value $X=x$. The conditional expected log-loss is

$$
\mathbb{E}_{Y\sim p(\cdot\mid x)}[-\log p_\theta(Y\mid x)].
$$

This quantity has a name: it is the **cross-entropy** between the true conditional distribution $p(\cdot\mid x)$ and the model conditional distribution $p_\theta(\cdot\mid x)$:

$$
H\left(p(\cdot\mid x),p_\theta(\cdot\mid x)\right)
=\mathbb{E}_{Y\sim p(\cdot\mid x)}[-\log p_\theta(Y\mid x)].
$$

The interpretation is direct. Outcomes are drawn from the true conditional distribution, but the log-loss is computed using the model conditional distribution.

For a single outcome $Y=y$, the quantity

$$
-\log p_\theta(y\mid x)
$$

is the model's **surprise** at observing $y$ given $x$. Cross-entropy is the average surprise under the true conditional distribution.


## Entropy and Excess Log-Loss

The best possible log-loss at a fixed $x$ would use the true conditional distribution itself:

$$
H(p(\cdot\mid x))
=\mathbb{E}_{Y\sim p(\cdot\mid x)}[-\log p(Y\mid x)].
$$

This is the **conditional entropy** at $x$. It is the irreducible uncertainty in $Y$ given $X=x$. A model cannot make this term smaller, because it is determined by the true data-generating distribution.

The model's conditional cross-entropy can be decomposed as

$$
H\left(p(\cdot\mid x),p_\theta(\cdot\mid x)\right)
=H(p(\cdot\mid x))
+
\operatorname{KL}\left(p(\cdot\mid x)\|p_\theta(\cdot\mid x)\right).
$$

The second term is the **KL divergence** from the true conditional distribution to the model conditional distribution. It is the excess average log-loss from using $p_\theta(\cdot\mid x)$ instead of the true $p(\cdot\mid x)$.

Equivalently, for two distributions $p$ and $q$,

$$
\operatorname{KL}(p\|q)
=\mathbb{E}_{Y\sim p}\left[\log\frac{p(Y)}{q(Y)}\right]
=H(p,q)-H(p).
$$

Important details:

- $\operatorname{KL}(p\|q)\ge 0$.
- $\operatorname{KL}(p\|q)=0$ if and only if $p=q$.
- KL is not symmetric: usually $\operatorname{KL}(p\|q)\ne \operatorname{KL}(q\|p)$.
- If $q(y)=0$ where $p(y)>0$, then the KL divergence is infinite.


## The Main Payoff: Population NLL Minimizes Conditional KL

Now average the fixed-$x$ decomposition over $X$:

$$
\mathbb{E}_{(X,Y)\sim p}[-\log p_\theta(Y\mid X)]
=\mathbb{E}_X[H(p(\cdot\mid X))]
+
\mathbb{E}_X\left[
\operatorname{KL}\left(p(\cdot\mid X)\|p_\theta(\cdot\mid X)\right)
\right].
$$

The first term does not depend on $\theta$. Therefore, minimizing population NLL is equivalent to minimizing

$$
\mathbb{E}_X\left[
\operatorname{KL}\left(p(\cdot\mid X)\|p_\theta(\cdot\mid X)\right)
\right].
$$

This is the population meaning of NLL. When an ERM loss is NLL, the target is not merely a small training loss; it is a conditional distribution $p_\theta(\cdot\mid X)$ that is close to the true conditional distribution $p(\cdot\mid X)$ in expected KL divergence.

## Example: Bernoulli Cross-Entropy

For a Bernoulli random variable, let the true probability be $p=P(Y=1)$ and the model probability be $q$. Then

$$
H(p,q)=-p\log q-(1-p)\log(1-q),
$$

and

$$
H(p)=-p\log p-(1-p)\log(1-p).
$$

Thus

$$
\operatorname{KL}(p\|q)=H(p,q)-H(p).
$$

This is the distribution-level version of the binary cross-entropy expression used in logistic regression. In logistic regression, the model probability is

$$
q=p_w(Y=1\mid X=x)=\sigma(w^\top x).
$$


In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def plot_bernoulli_kl(p=0.7):
    q_grid = np.linspace(0.001, 0.999, 500)

    H_p = -p * np.log(p) - (1 - p) * np.log(1 - p)
    H_pq = -p * np.log(q_grid) - (1 - p) * np.log(1 - q_grid)
    KL = H_pq - H_p

    plt.figure(figsize=(8, 5))
    plt.plot(q_grid, H_pq, label="Cross-entropy H(p,q)")
    plt.plot(q_grid, KL, label="KL(p||q)")
    plt.axhline(H_p, linestyle="--", label="Entropy H(p)")
    plt.axvline(p, linestyle="--", label=f"true p = {p:.2f}")

    plt.xlabel("model probability q")
    plt.ylabel("value")
    plt.title(f"Bernoulli case with true probability p = {p:.2f}")
    plt.legend()
    plt.show()

interact(
    plot_bernoulli_kl,
    p=widgets.FloatSlider(value=0.7, min=0.01, max=0.99, step=0.01, description="p")
)


## Example: Gaussian KL Direction Matters

KL is not symmetric. For normal distributions,

$$
\operatorname{KL}\left(\mathcal{N}(\mu_p,\sigma_p^2)\|\mathcal{N}(\mu_q,\sigma_q^2)\right)
=\log\frac{\sigma_q}{\sigma_p}
+
\frac{\sigma_p^2+(\mu_p-\mu_q)^2}{2\sigma_q^2}
-
\frac{1}{2}.
$$

The code below compares $\operatorname{KL}(p\|q)$ with $\operatorname{KL}(q\|p)$ for two normal distributions. The asymmetry matters because NLL leads specifically to $\operatorname{KL}(p\|p_\theta)$, where the expectation is taken under the true data-generating distribution.


In [ ]:
rng = np.random.default_rng(17)

mu_ref = rng.normal(1)
sigma_ref = abs(rng.normal(1)) + 0.2

def normal_pdf(x, mu, sigma):
    return (1.0 / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def kl_gaussian(mu_p, sigma_p, mu_q, sigma_q):
    return np.log(sigma_q / sigma_p) + (
        sigma_p**2 + (mu_p - mu_q)**2
    ) / (2 * sigma_q**2) - 0.5

def plot_normals(mu=1.0, sigma=1.5):
    x_min = min(mu_ref - 4 * sigma_ref, mu - 4 * sigma) - 1
    x_max = max(mu_ref + 4 * sigma_ref, mu + 4 * sigma) + 1
    x = np.linspace(x_min, x_max, 1000)

    p_density = normal_pdf(x, mu_ref, sigma_ref)
    q_density = normal_pdf(x, mu, sigma)

    kl_pq = kl_gaussian(mu_ref, sigma_ref, mu, sigma)
    kl_qp = kl_gaussian(mu, sigma, mu_ref, sigma_ref)

    plt.figure(figsize=(8, 5))
    plt.plot(x, p_density, label="reference p")
    plt.plot(x, q_density, label=fr"model q: N({mu:.2f}, {sigma:.2f}^2)")
    plt.xlabel("x")
    plt.ylabel("density")
    plt.title("Two normal densities")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

    print(f"KL(reference || model) = {kl_pq:.6f}")
    print(f"KL(model || reference) = {kl_qp:.6f}")

interact(
    plot_normals,
    mu=widgets.FloatSlider(value=1.0, min=-4.0, max=4.0, step=0.01, description="mu"),
    sigma=widgets.FloatSlider(value=1.5, min=0.2, max=4.0, step=0.01, description="sigma")
)


## Summary

ERM is a general framework: choose a loss and minimize its empirical average. Conditional MLE is a probabilistic framework: choose parameters that make the observed labels likely given the inputs. These frameworks coincide when the loss is the negative log-likelihood for a conditional model.

Under this choice, familiar losses acquire probabilistic interpretations. Squared error is the NLL for a Gaussian conditional model with fixed variance. Logistic cross-entropy is the NLL for a Bernoulli conditional model.

The population interpretation is the final link. Expected NLL is conditional cross-entropy: the average log-loss when future labels are generated from the true conditional distribution but scored under the model. This equals irreducible conditional entropy plus expected conditional KL. Since the entropy term does not depend on the model, minimizing population NLL fits the conditional distribution in expected KL divergence.


## Review Questions

See: @sec-nll-questions.